Carga del Dataset y preparación de los datos

In [10]:
import pandas as pd

#Definimos la URL del archivo CSV
url = "https://raw.githubusercontent.com/brunopless/TFM-Equipo-4-BDDS/main/entrega-3/df_bcn_imputado.csv"

#Leemos los datos directamente desde la URL
data = pd.read_csv(url)

#Mostramos las primeras filas del dataset para verificar que se cargó correctamente
print(data.head())

   Unnamed: 0  floor   price operation   size  exterior  rooms  bathrooms  \
0           1    4.0   749.0      rent   45.0         1      1          1   
1           2    2.0  4400.0      rent  170.0         1      3          4   
2           3    5.0  1700.0      rent   74.0         1      2          1   
3           4    1.0  2369.0      rent  127.0         1      2          2   
4           5    0.0  4000.0      rent   55.0         1      1          1   

    latitude  longitude  ...  numero_buenasmigas_cercanos_mas_de_cero  \
0  41.383719   2.175998  ...                                        1   
1  41.397182   2.142885  ...                                        0   
2  41.405040   2.211389  ...                                        0   
3  41.408746   2.215716  ...                                        0   
4  41.392297   2.169440  ...                                        0   

   numero_clubesnocturnos_cercanos_mas_de_cero  \
0                                            0  

In [11]:
import pandas as pd

#Separamos variables independientes y variable dependiente
X = data[['floor', 'size', 'exterior', 'rooms', 'bathrooms', 'newDevelopment', 'hasLift', 'topNewDevelopment',
    'numero_bancos_cercanos', 'numero_starbucks_cercanos', 'numero_atractivos_cercanos', 'numero_buenasmigas_cercanos',
    'numero_clubesnocturnos_cercanos', 'numero_mcdonalds_cercanos', 'numero_metro_cercanos', 'numero_parking_cercanos',
    'numero_parkingbici_cercanos', 'numero_parques_cercanos', 'numero_playas_cercanos', 'numero_santagloria_cercanos',
    'numero_vivari_cercanos', 'num_airbnbs_500', 'media_precios_airbnbs_500',
    'propertyType_chalet', 'propertyType_duplex', 'propertyType_flat', 'propertyType_penthouse', 'status_good',
    'status_newdevelopment', 'num_airbnbs_500_mas_de_cero', 'media_precios_airbnbs_500_mas_de_cero',
    'numero_bancos_cercanos_mas_de_cero', 'numero_starbucks_cercanos_mas_de_cero', 'numero_atractivos_cercanos_mas_de_cero',
    'numero_buenasmigas_cercanos_mas_de_cero', 'numero_clubesnocturnos_cercanos_mas_de_cero', 'numero_mcdonalds_cercanos_mas_de_cero',
    'numero_metro_cercanos_mas_de_cero', 'numero_parking_cercanos_mas_de_cero', 'numero_parkingbici_cercanos_mas_de_cero',
    'numero_parques_cercanos_mas_de_cero', 'numero_playas_cercanos_mas_de_cero', 'numero_santagloria_cercanos_mas_de_cero',
    'numero_vivari_cercanos_mas_de_cero']]
y = data['price_m2']


División datos para entrenamiento y prueba

In [12]:
from sklearn.model_selection import train_test_split

#Dividimos el dataset en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Escalamiento de variables

In [13]:
from sklearn.preprocessing import StandardScaler

#Definimos el escalador
scaler = StandardScaler()

#Datos de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)

#Datos de prueba
X_test_scaled = scaler.transform(X_test)

Entrenamiento modelo SVR:

In [14]:
from sklearn.svm import SVR

#Creamos el modelo SVR con el kernel RBF
svr_model = SVR(kernel='rbf')

#Entrenamos el modelo
svr_model.fit(X_train_scaled, y_train)

SVR()

Predicciones

In [15]:
#Predecimos los precios en los datos de prueba
y_pred = svr_model.predict(X_test_scaled)

Evaluación del modelo

In [16]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
import numpy as np
#MSE
mse = mean_squared_error(y_test, y_pred)

#R²
r2 = r2_score(y_test, y_pred)

#R² ajustado
n = len(y_test)
p = X_test.shape[1]
r2_ajustado = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f'MSE: {mse}')
print(f'R²: {r2}')
print(f'R² ajustado: {r2_ajustado}')

#MAE
mae = mean_absolute_error(y_test, y_pred)

#MAPE
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f'MAE: {mae}')
print(f'MAPE: {mape:.2f}%')

#Suma de Residuos al Cuadrado (RSS)
rss = np.sum((y_test - y_pred) ** 2)

#AIC
aic = n * np.log(rss / n) + 2 * p

print(f'AIC: {aic}')

MSE: 105.15048412578975
R²: 0.25271979838309355
R² ajustado: 0.24330659229344453
MAE: 7.190475700828856
MAPE: 24.85%
AIC: 16558.778687227725


Ajuste de Hiperparametros

In [17]:
from sklearn.model_selection import GridSearchCV

#Definimos los parámetros a ajustar
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

#Crear el modelo de búsqueda utilizando un grid
grid = GridSearchCV(SVR(), param_grid, refit=True, verbose=2)

#Entrenamos el modelo para con de hiperparámetros
grid.fit(X_train_scaled, y_train)

#Mejor combinación de parámetros para el modelo
print(grid.best_params_)

#Selección del mejor modelo para hacer predicciones
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)


Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.6s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.6s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.5s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.4s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  16.5s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.2s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  11.9s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.7s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.6s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.5s
[CV] END ......................C=0.1, gamma=0.01, kernel=rbf; total time=  12.8s
[CV] END ......................C=0.1, gamma=0.01

In [18]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import numpy as np

# Definir los parámetros a ajustar
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

#Creamos el modelo de búsqueda en cuadrícula
grid = GridSearchCV(SVR(), param_grid, refit=True, verbose=2)

#Entrenamos el modelo con búsqueda de hiperparámetros
grid.fit(X_train_scaled, y_train)

#Mejoramos combinación de parámetros
print("Mejores parámetros encontrados:", grid.best_params_)


best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)

#Métricas
r2 = r2_score(y_test, y_pred_best)
mae = mean_absolute_error(y_test, y_pred_best)
mape = mean_absolute_percentage_error(y_test, y_pred_best)

#Calculamos el R2 ajustado
n = len(y_test)
p = X_test_scaled.shape[1]
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

#Calculamos AIC

rss = np.sum((y_test - y_pred_best) ** 2)
aic = n * np.log(rss / n) + 2 * p

# Imprimir métricas
print("R2:", r2)
print("R2 Ajustado:", r2_adj)
print("MAE:", mae)
print("MAPE:", mape)
print("AIC:", aic)


Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.5s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.3s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.5s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  15.2s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  16.7s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  11.9s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.5s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.6s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  12.5s
[CV] END ......................C=0.1, gamma=0.01, kernel=rbf; total time=  12.6s
[CV] END ......................C=0.1, gamma=0.01